In [1]:
import pandas as pd
import warnings

# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
def backtest_trades(price_data, signal_data, entry_time_offset=None,
                    open_order_elimination=None, ignore_time_interval_before=None,
                    ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])

    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []
    initial_drawdown = 0  # For initial drawdown calculation
    nav_history = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        tp = row.get('TP')  # Fetch TP for this signal
        sl = row.get('SL')  # Fetch SL for this signal
        percentage_change = row.get('Percentage_Change')  # Fetch percentage change for this signal

        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'

        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']

        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''

        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 2:
                result = 'Ignored'
                reason = '2 open trades'
                ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(
                minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(
                minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Ignored',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': 'Signal around economic event'
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, open_order_elimination, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # Update NAV on filled order
        current_margin *= (1 - 0.0002)

        if side == 'Buy':
            tp_price = entry_price * (1 + tp) if tp is not None else None
            sl_price = entry_price * (1 - sl) if sl is not None else None
        else:
            tp_price = entry_price * (1 - tp) if tp is not None else None
            sl_price = entry_price * (1 + sl) if sl is not None else None

        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (
                                side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (
                                entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (
                                exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss',
                          'ended before data with no exact price']:

            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        # Calculate initial drawdown
        if current_margin < 100000:
            drawdown = ((100000 - current_margin) / 100000) * 100
            initial_drawdown = max(initial_drawdown, drawdown)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin       
        initial_margin = current_margin

        nav_history.append(nav)
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])

        output_data = pd.concat([output_data, new_row], ignore_index=True)
  
    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])

    # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000

    for date, nav in daily_nav.items():
        daily_return = ((nav - previous_day_nav) / previous_day_nav) * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)
    
    # Calculate Monthly Max Drawdown
    output_data['Month'] = output_data['Datetime'].dt.to_period('M')
    monthly_max_drawdowns = {}

    for month, group in output_data.groupby('Month'):
        peak_nav = group['NAV'].iloc[0]  # Start with the first NAV of the month
        max_drawdown_in_month = 0
        local_peak = peak_nav

        for nav in group['NAV']:
            if nav > local_peak:
                local_peak = nav  # Update the peak if a new high is found
            else:
                # Calculate drawdown from the peak to the current NAV
                drawdown = ((local_peak - nav) / local_peak) * 100
                max_drawdown_in_month = max(max_drawdown_in_month, drawdown)  # Track the maximum drawdown

        monthly_max_drawdowns[month] = max_drawdown_in_month


    output_data['Monthly Max Drawdown'] = output_data['Month'].map(monthly_max_drawdowns)
    output_data['Initial Drawdown'] = initial_drawdown
   
    output_data.drop(columns=['Month'], inplace=True)
    
    return output_data


In [3]:
# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"


def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break

    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'

    return result, duration_str


def determine_entry(price_data, signal_datetime, percentage_change, side, open_order_elimination, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None

    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (
                1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)

    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=open_order_elimination)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [20]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\filtered_signals_with_optimized_values.csv',
                            parse_dates=['Datetime'])

month = 7  # January (you can change this to the desired month)
year = 2024  # You can change this to the desired year
# # # 
# # # # Filter the signal data for the specified month and year
signal_data = signal_data[(signal_data['Datetime'].dt.month == month) & (signal_data['Datetime'].dt.year == year)]

In [21]:
result = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    entry_time_offset=0,  # For example, 5 minutes offset
    open_order_elimination=120,  # Number of open orders to eliminate signal
    ignore_time_interval_before=1020,  # Minutes before event to ignore signal
    ignore_time_interval_after=0  # Minutes after event to ignore signal
)
result

,Datetime,Side,Signal Open Price,Entry Price,TP Price,SL Price,Result,Duration,Execution Latency,ROI,NAV,Ignore Reason,Daily Return,Monthly Max Drawdown,Initial Drawdown
0,2024-07-02 09:00:00,Sell,62666.1,62691.16644,61876.181276,63443.460437,1,06:27:00,00:07:00,1.22910,101229.100130,,1.229100,0.738193,0
1,2024-07-04 09:00:00,Buy,57674.1,57622.19331,57910.304277,56873.104797,1,01:50:00,00:03:00,0.42966,101664.041132,,0.429660,0.738193,0
2,2024-07-05 09:00:00,Sell,54065.9,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,101664.041132,Signal around economic event,0.000000,0.738193,0
3,2024-07-06 11:00:00,Buy,56750.0,56732.97500,57470.503675,56392.577150,1,05:19:00,00:01:00,1.22910,102913.593994,,1.229100,0.738193,0
4,2024-07-07 01:00:00,Buy,57910.1,57834.81687,58413.165039,57603.477603,1,01:34:00,00:25:00,0.92931,103869.980417,,2.169832,0.738193,0
5,2024-07-07 09:00:00,Sell,57624.7,57636.22494,56886.954016,57924.406065,1,04:17:00,00:03:00,1.22910,105146.646482,,2.169832,0.738193,0
6,2024-07-07 13:00:00,Sell,57322.6,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,105146.646482,One open trade with the same side,2.169832,0.738193,0
7,2024-07-08 02:00:00,Buy,54930.0,54908.02800,55621.832364,54304.039692,1,02:46:00,00:00:00,1.22910,106439.004050,,1.229100,0.738193,0
8,2024-07-09 06:00:00,Buy,57261.8,57181.63348,57924.994715,56895.725313,1,08:12:00,00:02:00,1.22910,107747.245987,,1.229100,0.738193,0
9,2024-07-10 01:00:00,Buy,57467.9,NaN,NaN,NaN,Ignored,00:00:00,00:00:00,0.00000,107747.245987,Signal around economic event,0.000000,0.738193,0
